In [5]:
import pandas as pd

merged_dataset = pd.read_csv("merged_dataset_with_representation.csv", index_col=0)
merged_dataset = merged_dataset.drop_duplicates(subset=["molecule ID"])
merged_dataset.reset_index(drop=True, inplace=True)
merged_dataset

,molecule ID,evidence,Binding,kfold_0,kfold_1,kfold_2,kfold_3,kfold_4,smiles,sequence
0,CHEBI:35681,exp,1.0,valid,train,train,train,train,CC(C)O,MSIPSSQYGFVFNKQSGLNLRNDLPVHKPKAGQLLLKVDAVGLCHS...
1,CHEBI:30616,NaN,0.0,valid,train,train,train,train,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])OP(=O...,MSIPSSQYGFVFNKQSGLNLRNDLPVHKPKAGQLLLKVDAVGLCHS...
2,CHEBI:58349,exp,1.0,valid,train,train,train,train,N=C([O-])c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(...,MSIPSSQYGFVFNKQSGLNLRNDLPVHKPKAGQLLLKVDAVGLCHS...
3,C00019,NaN,0.0,valid,train,train,train,train,C[S+](CC[C@H](N)C(=O)[O-])C[C@H]1O[C@@H](n2cnc...,MSIPSSQYGFVFNKQSGLNLRNDLPVHKPKAGQLLLKVDAVGLCHS...
4,CHEBI:59789,NaN,0.0,valid,train,train,train,train,C[S@@+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3...,MSIPSSQYGFVFNKQSGLNLRNDLPVHKPKAGQLLLKVDAVGLCHS...
...,...,...,...,...,...,...,...,...,...,...
1346,CHEBI:58391,NaN,0.0,train,train,train,valid,train,CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C/CC/C(C)=C...,MSFPFASLLKRPSAISSLLSLKKPGSWSSILLKAVGVLSRDSRWHS...
1347,C03701,NaN,0.0,train,train,valid,train,train,*C(=O)NC(CO[C@@H]1O[C@H](CO)[C@@H](O)[C@H](O)[...,MKTQVAIIGAGPSGLLLGQLLHKAGIDNVILERQTPDYVLGRIRAG...
1348,CHEBI:36702,NaN,0.0,train,train,valid,train,train,COC[C@H](COP(=O)([O-])OCC[N+](C)(C)C)OC=O,MKRLLTLAWFLACSVPAVPGGLLELKSMIEKVTGKNAVKNYGFYGC...
1349,C03808,NaN,0.0,train,valid,train,train,train,*NC(=O)[C@H](COP(=O)(O)O)NC(*)=O,MVLEVPSITPGELHDLMRLHQDAEWPECKKMFPWAHDISFGQPPDF...


In [6]:
merged_dataset.to_csv("unique_compounds.csv", index=False)

In [8]:
import os
from deepmol.loaders import CSVLoader
from deepmol.compound_featurization import ThreeDimensionalMoleculeGenerator

# Processing parameters
timeout = 200
threads = 50
n_conformations = 1
max_iterations = 100
etkdg_version = 3
mode = "MMFF94"

dataset = CSVLoader("unique_compounds.csv", id_field="molecule ID", smiles_field="smiles").create_dataset()
            
# Generate 3D conformers
generator = ThreeDimensionalMoleculeGenerator(
    timeout_per_molecule=timeout, threads=threads,
    n_conformations=n_conformations, max_iterations=max_iterations
)
generator.generate(dataset, etkdg_version=etkdg_version, mode=mode)

# Save as SDF
output_sdf_path = os.path.join("unique_compounds_conformers.sdf")
dataset.to_sdf(output_sdf_path)

/home/jcapela/fingerprints_benchmark/DeepMol/src/deepmol/compound_featurization/__init__.py:20: UserWarning: Mol2Vec not available. Please install it to use it. (pip install git+https://github.com/samoturk/mol2vec#egg=mol2vec)
  warnings.warn("Mol2Vec not available. Please install it to use it. "
Generating 3D structure: 100%|██████████| 1351/1351 [05:46<00:00,  3.90it/s]


In [9]:
from deepmol.loaders import SDFLoader

compounds_dataset = SDFLoader("unique_compounds_conformers.sdf", id_field="_ID").create_dataset()

In [ ]:
from deepmol.compound_featurization import All3DDescriptors, MixedFeaturizer, TwoDimensionDescriptors

MixedFeaturizer([All3DDescriptors(mandatory_generation_of_conformers=False), TwoDimensionDescriptors()]).featurize(compounds_dataset, inplace=True)

In [12]:
compounds_dataset.get_shape()

2025-05-07 17:55:24,989 — INFO — Mols_shape: (1287,)
2025-05-07 17:55:24,990 — INFO — Features_shape: (1287, 849)
2025-05-07 17:55:24,991 — INFO — Labels_shape: None


((1287,), (1287, 849), None)

In [14]:
compounds_dataset.to_sdf(path="unique_compounds_with_features.sdf")
compounds_dataset.to_csv("unique_compounds_with_features.csv", index=False)

In [5]:
import pandas as pd
compounds_with_features = pd.read_csv("unique_compounds_with_features.csv")
compounds_with_features

,ids,smiles,Asphericity,AUTOCORR3D_0,AUTOCORR3D_1,AUTOCORR3D_2,AUTOCORR3D_3,AUTOCORR3D_4,AUTOCORR3D_5,AUTOCORR3D_6,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,CHEBI:35681,CC(C)O,0.212554,0.744,1.221,0.000,0.000,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,CHEBI:30616,Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO[P@@](=O)([O-])O...,0.727682,0.104,0.276,0.357,0.488,0.461,0.592,0.536,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,CHEBI:58349,N=C([O-])c1ccc[n+]([C@@H]2O[C@H](CO[P@@](=O)(O...,0.523713,0.067,0.173,0.235,0.312,0.376,0.425,0.453,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,C00019,C[S@+](CC[C@H](N)C(=O)[O-])C[C@H]1O[C@@H](n2cn...,0.474330,0.121,0.293,0.416,0.472,0.525,0.530,0.501,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,CHEBI:59789,C[S@+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c...,0.762966,0.121,0.292,0.415,0.474,0.539,0.592,0.587,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,CHEBI:58821,CC1=CCC[C@H]2[C@](C)(CC/C(C)=C/CO[P@@](=O)([O-...,0.685736,0.113,0.297,0.422,0.518,0.476,0.490,0.482,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1283,CHEBI:64283,C/C(=C\CO[P@](=O)([O-])OP(=O)([O-])[O-])CC[C@@...,0.544066,0.110,0.295,0.400,0.523,0.424,0.514,0.401,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1284,CHEBI:58391,CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C/CC/C(C)=C...,0.921135,0.055,0.119,0.150,0.199,0.215,0.288,0.272,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1285,CHEBI:36702,COC[C@H](CO[P@@](=O)([O-])OCC[N+](C)(C)C)OC=O,0.235513,0.156,0.349,0.386,0.488,0.584,0.708,0.476,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [2]:
"CHEBI:17245" in compounds_with_features

False

In [6]:
from plants_sm.io.pickle import write_pickle

features_dict = compounds_with_features.set_index('ids').drop(columns='smiles').apply(lambda row: row.to_numpy(), axis=1).to_dict()
write_pickle("compounds_features.pkl", features_dict)

True

# Filter datapoints only compounds with structure and features

In [7]:
import pandas as pd

train_dataset = pd.read_csv("train_dataset_w_representation.csv")
train_dataset.shape

(53406, 14)

In [8]:
train_dataset = train_dataset[train_dataset["molecule ID"].isin(compounds_with_features["ids"])]
train_dataset.shape

(51160, 14)

In [9]:
train_dataset.to_csv("train_dataset_w_representation_filtered.csv", index=False)

In [10]:
import pandas as pd

test_dataset = pd.read_csv("test_dataset_w_representation.csv")
test_dataset.shape

(12762, 14)

In [11]:
test_dataset = test_dataset[test_dataset["molecule ID"].isin(compounds_with_features["ids"])]
test_dataset.shape
test_dataset.to_csv("test_dataset_w_representation_filtered.csv", index=False)

In [12]:
test_dataset.shape

(12336, 14)